In [1]:

print("Task 03 - Features and Tuning")
print("Machine Learning Project Started Successfully")

Task 03 - Features and Tuning
Machine Learning Project Started Successfully


In [3]:

import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the dataset
data = load_breast_cancer()

# Convert dataset into a DataFrame
df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

# Add the target column
df["target"] = data.target

# Display the first five rows
print(df.head())



   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [5]:

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDataset Information:")
df.info()



Dataset Shape: (569, 31)

Column Names:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'target']

Missing Values:
0

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean pe

In [6]:
# Feature Engineering

# Create the first new feature
df["radius_texture_ratio"] = (
    df["mean radius"] / (df["mean texture"] + 1e-6)
)

# Create the second new feature
df["area_perimeter_ratio"] = (
    df["mean area"] / (df["mean perimeter"] + 1e-6)
)

# Display the new features
print(df[[
    "radius_texture_ratio",
    "area_perimeter_ratio"
]].head())

print("Feature Engineering Completed!")

   radius_texture_ratio  area_perimeter_ratio
0              1.733140              8.151466
1              1.157569              9.977427
2              0.926588              9.253846
3              0.560353              4.976798
4              1.414923              9.600296
Feature Engineering Completed!


In [7]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Data splitting completed!")

Training data: (455, 32)
Testing data: (114, 32)
Data splitting completed!


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Create the baseline model
model = RandomForestClassifier(random_state=42)

# Perform 5-fold cross-validation
cv_scores_before = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores before tuning:")
print(cv_scores_before)

print("Mean accuracy before tuning:",
      cv_scores_before.mean())

Cross-validation scores before tuning:
[0.95604396 0.97802198 0.93406593 0.94505495 0.96703297]
Mean accuracy before tuning: 0.956043956043956


In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameters for tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train and tune the model
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best Cross-Validation Accuracy: 0.9604395604395606


In [10]:
import pandas as pd

# Calculate mean accuracy before and after tuning
before_accuracy = cv_scores_before.mean()
after_accuracy = grid_search.best_score_

# Create comparison table
comparison = pd.DataFrame({
    "Stage": ["Before Tuning", "After Tuning"],
    "Mean Cross-Validation Accuracy": [
        before_accuracy,
        after_accuracy
    ]
})

print(comparison)

# Calculate improvement
improvement = (after_accuracy - before_accuracy) * 100

print(f"\nImprovement: {improvement:.2f} percentage points")

           Stage  Mean Cross-Validation Accuracy
0  Before Tuning                        0.956044
1   After Tuning                        0.960440

Improvement: 0.44 percentage points


In [11]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on test data
y_pred = best_model.predict(X_test)

# Calculate test accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print("Final Test Accuracy:", test_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=data.target_names
    )
)

Final Test Accuracy: 0.956140350877193

Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## Conclusion

In this project, a machine learning model was developed using the Breast Cancer dataset. Feature engineering was performed by creating two new features. A Random Forest Classifier was used as the baseline model, and five-fold cross-validation was applied to evaluate its performance.

GridSearchCV was used for hyperparameter tuning to identify suitable model parameters. The model's performance before and after tuning was compared using cross-validation accuracy. Finally, the tuned model was evaluated on the test dataset.

The results show how feature engineering and hyperparameter tuning can help evaluate and improve a machine learning model. The actual performance improvement was determined from the experimental results.
